## Solution for Assignment 02 Question 3

### Setup

In [9]:
import torch
import torch.nn as nn
from torchvision.models import resnet18
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

### CONSTANTS ### 
BATCH_SIZE = 256
EPOCHS = 10
LR = 1e-3

### MODEL ### 
# Initialize the ResNet18 model with random weights using torchvision
model = resnet18(weights=None)
# Adjust input and output layers to work with FMNIST
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Linear(model.fc.in_features, 10)

In [10]:
### DATA ###

# Transform to convert FMNIST data into normalized tensors
fmnist_data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Init train and test datasets
train_data = datasets.FashionMNIST(
    root='/datasets/FMNIST/',
    train=True,
    download=True,
    transform=fmnist_data_transform
)

test_data = datasets.FashionMNIST(
    root='/datasets/FMNIST/',
    train=False,
    download=True,
    transform=fmnist_data_transform
)

# Define dataloaders
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

### a)

In [11]:
### TRAIN and TEST ###

optim = torch.optim.Adam(
    params=model.parameters(),
    lr=LR,
    weight_decay=0
)
criterion = nn.CrossEntropyLoss()

LAMBDA = 0.0001


for epoch in range(EPOCHS):
    model.train()

    for features, labels in train_loader:
        # Calculate original cross entropy loss
        output_model = model(features)
        loss = criterion(output_model, labels)
        # Add regularization term
        loss += LAMBDA * sum((p**2).sum() for p in model.parameters()) / 2

        # Backpropagate
        optim.zero_grad()
        loss.backward()
        optim.step()

    model.eval()

    total = 0
    correct = 0
    for features, labels in test_loader:
        output_model_test = model(features)
        predictions = output_model_test.argmax(axis=1)
        total += output_model_test.size(0)
        correct += (predictions == labels).sum().item()

    test_acc = correct / total
    print(f"Epoch {epoch +1}: Test accuracy = {test_acc:.4f}")

KeyboardInterrupt: 